# Herlev Kaggle Fast Training Notebook

A faster preset for training the Herlev-only model on Kaggle.

Use this when you want a quicker single-split run. Set `USE_KFOLD = True` in the config cell if you want 5-fold training instead.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']


def find_training_script(repo_root: Path) -> Path | None:
    candidate = repo_root / 'backend' / 'train.py'
    return candidate if candidate.exists() else None


def discover_code_root() -> Path | None:
    search_roots = [WORK_DIR, Path('/kaggle/input')]
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for candidate in search_root.rglob('train.py'):
            if candidate.is_file() and candidate.parent.name == 'backend':
                return candidate.parent.parent
    for candidate in [WORK_DIR / 'Cervical-Cancer-Classifier', WORK_DIR / 'repo']:
        if find_training_script(candidate) is not None:
            return candidate
    return None


REPO_DIR = discover_code_root()
if REPO_DIR is None:
    git_url = os.environ.get('REPO_GIT_URL', 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier').strip()
    if not git_url:
        raise FileNotFoundError('Upload the refactored repo as a Kaggle dataset or set REPO_GIT_URL.')
    REPO_DIR = WORK_DIR / 'Cervical-Cancer-Classifier'
    subprocess.check_call(['git', 'clone', git_url, str(REPO_DIR)])

TRAIN_SCRIPT = find_training_script(REPO_DIR)
if TRAIN_SCRIPT is None:
    raise FileNotFoundError(
        f'Could not find backend/train.py under {REPO_DIR}. '\
        'Upload the refactored repo to Kaggle so the notebook can find it.'
    )

BACKEND_DIR = TRAIN_SCRIPT.parent if TRAIN_SCRIPT.name == 'train.py' else TRAIN_SCRIPT.parent.parent
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BACKEND_DIR / 'requirements.txt')])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

def is_dataset_root(root: Path) -> bool:
    if not root.exists() or not root.is_dir():
        return False
    if (root / 'train').exists() or (root / 'val').exists():
        return True
    return any((root / cls).exists() for cls in CLASS_NAMES)

DATA_CANDIDATES = [REPO_DIR / 'data', WORK_DIR / 'data']
if Path('/kaggle/input').exists():
    for candidate in Path('/kaggle/input').rglob('*'):
        if candidate.is_dir() and is_dataset_root(candidate):
            DATA_CANDIDATES.append(candidate)

DATA_DIR = next((candidate for candidate in DATA_CANDIDATES if is_dataset_root(candidate)), None)
if DATA_DIR is None:
    raise FileNotFoundError('No Herlev dataset found.')

print('Repo   :', REPO_DIR)
print('Script :', TRAIN_SCRIPT)
print('Backend:', BACKEND_DIR)
print('Data   :', DATA_DIR)


In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/checkpoints_fast')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BACKBONE = os.environ.get('BACKBONE', 'convnextv2_tiny')
EPOCHS = int(os.environ.get('EPOCHS', '35'))
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '12'))
IMAGE_SIZE = int(os.environ.get('IMAGE_SIZE', '224'))
VAL_SPLIT = float(os.environ.get('VAL_SPLIT', '0.2'))
WORKERS = int(os.environ.get('WORKERS', '2'))
SEED = int(os.environ.get('SEED', '42'))
LR = float(os.environ.get('LR', '5e-5'))
WEIGHT_DECAY = float(os.environ.get('WEIGHT_DECAY', '1e-4'))
PATIENCE = int(os.environ.get('PATIENCE', '10'))
ACCUMULATION_STEPS = int(os.environ.get('ACCUMULATION_STEPS', '1'))
LOSS_TYPE = os.environ.get('LOSS_TYPE', 'class_balanced_focal')
UNDERSAMPLE = os.environ.get('UNDERSAMPLE', 'random')
USE_KFOLD = os.environ.get('USE_KFOLD', '0') == '1'
K_FOLDS = int(os.environ.get('K_FOLDS', '5'))

print('Output dir       :', OUTPUT_DIR)
print('Backbone         :', BACKBONE)
print('Epochs           :', EPOCHS)
print('Batch size       :', BATCH_SIZE)
print('Image size       :', IMAGE_SIZE)
print('Use k-fold       :', USE_KFOLD)
print('K folds          :', K_FOLDS)
print('Undersample      :', UNDERSAMPLE)

In [ ]:
import os
import subprocess
import sys

cmd = [
    sys.executable, str(TRAIN_SCRIPT),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--image-size', str(IMAGE_SIZE),
    '--val-split', str(VAL_SPLIT),
    '--backbone', BACKBONE,
    '--undersample', UNDERSAMPLE,
    '--lr', str(LR),
    '--weight-decay', str(WEIGHT_DECAY),
    '--patience', str(PATIENCE),
    '--accumulation-steps', str(ACCUMULATION_STEPS),
    '--workers', str(WORKERS),
    '--seed', str(SEED),
    '--loss-type', LOSS_TYPE,
]
if USE_KFOLD:
    cmd.extend(['--use-kfold', '--k-folds', str(K_FOLDS)])

print('Running:')
print(' '.join(cmd))
print()
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in proc.stdout:
    print(line, end='')
code = proc.wait()
if code != 0:
    raise RuntimeError(f'Training failed with exit code {code}')
print('Training complete.')


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path('/kaggle/working/herlev_fast_checkpoints.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.glob('*')):
        if p.is_file() and p.suffix in {'.pt', '.json'}:
            zf.write(p, arcname=str(Path('backend') / 'Checkpoints' / p.name))

print('Zip created:', zip_path)
print('Extract this zip into your local repo root so the files land in backend/Checkpoints.')

## Quick Toggle

- Single split: leave `USE_KFOLD = False`
- 5-fold: set `USE_KFOLD = True`
- Faster run: keep `IMAGE_SIZE = 224` and `BACKBONE = 'convnextv2_tiny'`